# 中证800 V58：技术/价格特征消融实验

## 实验目的

这个 notebook 只测试一个问题：V46/V56 这套 direct LGB 模型里的技术/价格/量价特征，到底是在贡献可交易边际，还是主要制造 FP 和过拟合。

## 固定项

- 股票池：中证800 / `000906.XSHG`
- 标签：`alpha_1m`
- 模型：LightGBM regression
- 训练：固定 120 轮，不使用 early stopping
- 训练边界：默认严格复刻 V56/V46 `legacy_unsealed_q4`，按 `rebalance_date <= train_end` 选训练样本
- 组合：先按模型分数取 top30 候选池，再做行业约束 top6
- 评估：月度收益、同约束随机 top6 分位、FP/FN 错例归因

## 变量

- `full`：V56/V46 风格完整特征
- `light_technical_price`：保留较粗的月频动量/风险风格，去掉更细的价格/成交/技术形态特征
- `no_technical_price`：去掉技术/价格/量价/动量风险类特征，只保留基本面、质量、成长、杠杆、流动性风格和现金流时序特征

## 判断方式

如果 `light_technical_price` 或 `no_technical_price` 的同约束随机分位、回撤、FP_not_top20 明显更好，同时收益没有大幅下降，说明技术/价格特征存在过拟合风险；如果 `full` 仍明显更强，说明这些特征仍然提供了实际可交易边际。


In [ ]:
import os
import gc
import pickle
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", every=None):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc)
    if every is None:
        every = max(1, int((total or 100) / 20))

    def _gen():
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                if total is None:
                    print("%s %s" % (desc, i))
                else:
                    print("%s %s/%s" % (desc, i, total))
            yield item
    return _gen()

# =========================
# Config
# =========================
DATA_PATH = "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"
OUT_DIR = "csi800_ml_v58_technical_price_ablation_outputs"

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
BENCHMARK = "000906.XSHG"
MODEL_FAMILY = "v58_v46_direct_technical_price_ablation"
BOUNDARY_POLICY = "legacy_unsealed_q4"
# Match V56/V46 legacy_unsealed_q4 exactly: train rows are selected by rebalance_date,
# not by next_date. This intentionally allows the final train month label to cross the
# calendar cutoff, because that is how the current practical anchor was trained.
USE_LEGACY_UNSEALED_BOUNDARY = True

TRAIN_WINDOW_SPECS = [
    {"tag": "2019_2023", "train_start": "2019-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01"},
    {"tag": "2019_2024", "train_start": "2019-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01"},
]

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
TOP_N_CANDIDATES = 30
STOCK_NUM = 6
INDUSTRY_CAP_RATIO = 0.20

RANDOM_SIM_N = 3000
RANDOM_SEED = 42
REALIZED_TOP_N = 6
REALIZED_TOP20_N = 20
ATTR_TOP_FEATURE_N = 15
EXPORT_MODELS = True

os.makedirs(OUT_DIR, exist_ok=True)
print("output dir:", OUT_DIR)


## 特征分组

`light_technical_price` 不是简单等于 JQ-only：它保留月频更粗的 `momentum / Rank1M / sharpe_ratio_60 / Variance20 / beta`，但剔除更容易受价格序列细节、复权、成交口径影响的技术/量价特征。


In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio",
    "book_to_price_ratio",
    "earnings_yield",
    "sales_to_price_ratio",
    "cash_earnings_to_price_ratio",
    "earnings_to_price_ratio",
    "roe_ttm",
    "roa_ttm",
    "gross_profit_ttm",
    "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit",
    "ACCA",
    "growth",
    "net_working_capital",
    "operating_profit_per_share",
    "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share",
    "super_quick_ratio",
    "MLEV",
    "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio",
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "liquidity",
    "beta",
    "ATR6",
    "MFI14",
    "DAVOL10",
    "VOL10",
    "VMACD",
    "VOSC",
    "Skewness20",
    "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

FULL_FEATURE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

LIGHT_TECHNICAL_KEEP = [
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "beta",
]

TECHNICAL_PRICE_COLS = [
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "beta",
    "ATR6",
    "MFI14",
    "DAVOL10",
    "VOL10",
    "VMACD",
    "VOSC",
    "Skewness20",
    "Kurtosis20",
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_Rank1M_rank_chg_1m",
]

LIGHT_TECHNICAL_REMOVE = [
    c for c in TECHNICAL_PRICE_COLS
    if c not in LIGHT_TECHNICAL_KEEP
]

NO_TECHNICAL_REMOVE = list(TECHNICAL_PRICE_COLS)

FEATURE_VARIANTS = [
    {
        "feature_variant": "full",
        "description": "V46/V56 full hybrid-light feature stack",
        "candidate_cols": list(FULL_FEATURE_COLS),
    },
    {
        "feature_variant": "light_technical_price",
        "description": "keep coarse monthly momentum/risk style; remove fine price-volume/technical extras",
        "candidate_cols": [c for c in FULL_FEATURE_COLS if c not in LIGHT_TECHNICAL_REMOVE],
    },
    {
        "feature_variant": "no_technical_price",
        "description": "remove technical/price/momentum-risk family; keep fundamental/liquidity style and cashflow temporal feature",
        "candidate_cols": [c for c in FULL_FEATURE_COLS if c not in NO_TECHNICAL_REMOVE],
    },
]

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

variant_manifest_rows = []
for v in FEATURE_VARIANTS:
    cols = list(v["candidate_cols"])
    variant_manifest_rows.append({
        "feature_variant": v["feature_variant"],
        "candidate_feature_count": len(cols),
        "description": v["description"],
        "candidate_features": ",".join(cols),
        "removed_from_full": ",".join([c for c in FULL_FEATURE_COLS if c not in cols]),
    })
variant_manifest_df = pd.DataFrame(variant_manifest_rows)
display(variant_manifest_df[["feature_variant", "candidate_feature_count", "description", "removed_from_full"]])


## 基础 helper

这些 helper 保持保守 pandas 写法，避免空样本、NaN、除零、旧 pandas 兼容问题。


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col])
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def calc_drawdown_from_returns(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return np.nan
    nav = (1.0 + s).cumprod()
    dd = nav / nav.cummax() - 1.0
    return float(dd.min())


def summarize_return_series(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"months": 0, "cum_ret": np.nan, "mean_ret": np.nan, "win_rate": np.nan, "max_drawdown": np.nan}
    return {
        "months": int(len(s)),
        "cum_ret": float((1.0 + s).prod() - 1.0),
        "mean_ret": float(s.mean()),
        "win_rate": float((s > 0).mean()),
        "max_drawdown": calc_drawdown_from_returns(s),
    }


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []

    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)

    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    if len(cols) == 0:
        raise ValueError("no candidate feature exists in train data")
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    y = d[target_col].astype(float).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


def split_diag_valid(train_df):
    months = sorted(pd.to_datetime(train_df["rebalance_date"].dropna().unique()))
    if len(months) <= 8:
        return train_df.copy(), train_df.copy()
    n_valid = max(6, int(round(len(months) * 0.20)))
    valid_months = set(months[-min(n_valid, len(months) - 1):])
    fit = train_df[~train_df["rebalance_date"].isin(valid_months)].copy()
    valid = train_df[train_df["rebalance_date"].isin(valid_months)].copy()
    if fit.empty or valid.empty:
        return train_df.copy(), train_df.copy()
    return fit, valid


## 数据加载

本实验固定读取一个数据 CSV，避免隐式内存变量或候选路径导致口径漂移。


In [ ]:
def load_dataset(path):
    if not os.path.exists(path):
        raise IOError("data csv not found: " + path)
    df = pd.read_csv(path)
    df = safe_to_datetime(df, ["rebalance_date", "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = df["raw_return_1m"] - df["benchmark_csi800_1m"]
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "feature_date", "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    return df


def normalize_train_spec(spec):
    out = dict(spec)
    out["train_start"] = pd.Timestamp(out.get("train_start", "2019-01-01"))
    out["train_end"] = pd.Timestamp(out["train_end"])
    out["test_start"] = pd.Timestamp(out.get("test_start", out["train_end"] + pd.Timedelta(days=1)))
    if not out.get("tag"):
        out["tag"] = "{}_{}".format(out["train_start"].strftime("%Y%m%d"), out["train_end"].strftime("%Y%m%d"))
    return out


def make_train_df(df_all, spec):
    spec = normalize_train_spec(spec)
    mask = (df_all[DATE_COL] >= spec["train_start"]) & (df_all[DATE_COL] <= spec["train_end"])
    if not USE_LEGACY_UNSEALED_BOUNDARY:
        mask = mask & (df_all["next_date"] <= spec["train_end"])
    df = df_all[mask].copy()
    return df


def make_test_df(df_all, spec):
    spec = normalize_train_spec(spec)
    df = df_all[df_all[DATE_COL] >= spec["test_start"]].copy()
    return df


df_all = load_dataset(DATA_PATH)
print("loaded:", df_all.shape)
print(df_all[["rebalance_date", "feature_date", "next_date"]].agg(["min", "max"]))
print("target:", TARGET_COL)
display(df_all[[TARGET_COL]].describe())

available_rows = []
for v in FEATURE_VARIANTS:
    cols = [c for c in v["candidate_cols"] if c in df_all.columns]
    available_rows.append({
        "feature_variant": v["feature_variant"],
        "configured": len(v["candidate_cols"]),
        "available": len(cols),
        "missing": ",".join([c for c in v["candidate_cols"] if c not in df_all.columns]),
    })
available_df = pd.DataFrame(available_rows)
display(available_df)


## 模型训练和导出

bundle schema 兼容当前 V46 回测壳：加载一个 pkl，读取 `base_feature_cols/base_fill_values/base_model`，再由回测代码自动推断需要哪些特征。


In [ ]:
def train_direct_lgb(train_df, feature_cols):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, train_index = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix")
    model = lgb.train(
        params,
        lgb.Dataset(X_train, label=y_train),
        num_boost_round=max(1, int(FIXED_ITER)),
    )
    pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    train_rank_ic = safe_rank_ic(y_train, pred)
    return {
        "model": model,
        "fill_values": fill_values,
        "train_rows": int(len(X_train)),
        "train_rank_ic": train_rank_ic,
    }


def score_with_model(df, model, feature_cols, fill_values):
    X = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(fill_values).fillna(0)
    return np.asarray(model.predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)


def make_model_id(spec, feature_variant):
    spec = normalize_train_spec(spec)
    train_start_tag = spec["train_start"].strftime("%Y%m%d")
    train_end_tag = spec["train_end"].strftime("%Y%m%d")
    return "v58_{}_{}_start{}_cutoff{}_fixed{}".format(
        feature_variant,
        spec["tag"],
        train_start_tag,
        train_end_tag,
        int(FIXED_ITER),
    )


def needs_v4_adapter(feature_cols):
    adapter_cols = set(HYBRID_LIGHT_EXTRA_COLS)
    return any(c in adapter_cols for c in feature_cols)


def export_bundle(model_id, spec, feature_variant, trained, feature_cols, removed_cols, diag_rank_ic, train_df):
    model_file = "model_candidate_{}.pkl".format(model_id)
    out_path = os.path.join(OUT_DIR, model_file)
    bundle = {
        "objective": "v210_refit_fixed_iter_overlay",
        "research_version": model_id,
        "benchmark": BENCHMARK,
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "label_end": spec["train_end"],
        "boundary_policy": BOUNDARY_POLICY,
        "require_label_end_within_train": bool(not USE_LEGACY_UNSEALED_BOUNDARY),
        "legacy_unsealed_boundary": bool(USE_LEGACY_UNSEALED_BOUNDARY),
        "model_family": MODEL_FAMILY,
        "feature_variant": feature_variant,
        "target_col": TARGET_COL,
        "target_note": "V58 technical/price ablation: direct LGB alpha_1m, fixed iteration, no early stopping",
        "data_file": DATA_PATH,
        "protocol": "v58_v46_direct_feature_ablation_fixed_cutoff",
        "training_policy": "fixed_window_legacy_unsealed",
        "param_set": "v46_base_ff10_original",
        "base_params": dict(BASE_PARAMS_FF10),
        "base_model": trained["model"],
        "base_feature_cols": list(feature_cols),
        "base_fill_values": dict(trained["fill_values"]),
        "base_best_iter": int(FIXED_ITER),
        "model_iter": int(FIXED_ITER),
        "fixed_iter": int(FIXED_ITER),
        "base_inner_metrics": {
            "train_rank_ic": float(trained["train_rank_ic"]) if not pd.isnull(trained["train_rank_ic"]) else np.nan,
            "diag_rank_ic": float(diag_rank_ic) if not pd.isnull(diag_rank_ic) else np.nan,
        },
        "base_removed_features": list(removed_cols),
        "residual_model": None,
        "residual_feature_cols": [],
        "residual_fill_values": {},
        "overlay_weight": 0.0,
        "overlay_mode": "direct",
        "top_n_candidates": TOP_N_CANDIDATES,
        "stock_num": STOCK_NUM,
        "industry_cap_ratio": INDUSTRY_CAP_RATIO,
        "requires_v4_feature_adapter": bool(needs_v4_adapter(feature_cols)),
        "requires_industry_relative_adapter": False,
        "uses_time_weight": False,
        "uses_sample_weight": False,
        "uses_current_valid_for_training": False,
        "final_role": "v58_technical_price_ablation_candidate",
        "train_row_count": int(len(train_df)),
        "train_month_count": int(train_df[DATE_COL].nunique()),
        "max_train_rebalance_date": str(train_df[DATE_COL].max().date()),
        "max_train_next_date": str(train_df["next_date"].max().date()),
    }
    if EXPORT_MODELS:
        with open(out_path, "wb") as f:
            pickle.dump(bundle, f, protocol=2)
    return out_path, model_file


def train_one_variant_spec(df_all, variant, spec):
    spec = normalize_train_spec(spec)
    train_df = make_train_df(df_all, spec)
    if train_df.empty:
        raise ValueError("empty train_df for " + str(spec["tag"]))
    diag_fit_df, diag_valid_df = split_diag_valid(train_df)
    feature_cols, removed_cols = select_features_train_only(diag_fit_df, variant["candidate_cols"])
    trained = train_direct_lgb(train_df, feature_cols)
    X_valid, y_valid, _, _ = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, trained["fill_values"])
    valid_pred = np.asarray(trained["model"].predict(X_valid[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    diag_rank_ic = safe_rank_ic(y_valid, valid_pred)
    feature_variant = variant["feature_variant"]
    model_id = make_model_id(spec, feature_variant)
    model_path, model_file = export_bundle(model_id, spec, feature_variant, trained, feature_cols, removed_cols, diag_rank_ic, train_df)
    row = {
        "model_id": model_id,
        "model_family": MODEL_FAMILY,
        "feature_variant": feature_variant,
        "tag": spec["tag"],
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "train_rows": int(len(train_df)),
        "train_months": int(train_df[DATE_COL].nunique()),
        "max_train_next_date": str(train_df["next_date"].max().date()),
        "candidate_feature_count": len(variant["candidate_cols"]),
        "feature_count": len(feature_cols),
        "removed_feature_count": len(removed_cols),
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": diag_rank_ic,
        "model_file": model_file,
        "model_path": model_path,
        "feature_cols": ",".join(feature_cols),
        "removed_features": ",".join(removed_cols),
    }
    return row, trained, feature_cols


## 组合生成与月度评估

主组合严格按线上近似逻辑生成：`score top30 -> industry-neutral top6`。


In [ ]:
def build_industry_neutral_targets(sorted_stocks, industry_map, target_num, max_per_industry):
    known_industries = set([
        industry_map.get(stock, "UNKNOWN")
        for stock in sorted_stocks
        if industry_map.get(stock, "UNKNOWN") != "UNKNOWN"
    ])
    if len(known_industries) < 3:
        return sorted_stocks[:min(target_num, len(sorted_stocks))]

    selected = []
    industry_count = {}
    for stock in sorted_stocks:
        industry = industry_map.get(stock, "UNKNOWN")
        if industry == "UNKNOWN":
            continue
        if industry_count.get(industry, 0) == 0:
            selected.append(stock)
            industry_count[industry] = 1
            if len(selected) >= target_num:
                return selected

    for stock in sorted_stocks:
        if stock in selected:
            continue
        industry = industry_map.get(stock, "UNKNOWN")
        if industry == "UNKNOWN":
            continue
        cnt = industry_count.get(industry, 0)
        if cnt < max_per_industry:
            selected.append(stock)
            industry_count[industry] = cnt + 1
            if len(selected) >= target_num:
                return selected

    for stock in sorted_stocks:
        if stock not in selected:
            selected.append(stock)
            if len(selected) >= target_num:
                break
    return selected


def select_model_targets(month_df):
    sorted_all = list(month_df.sort_values("score", ascending=False)[STOCK_COL])
    candidate_stocks = sorted_all[:min(TOP_N_CANDIDATES, len(sorted_all))]
    candidate_df = month_df[month_df[STOCK_COL].isin(candidate_stocks)].copy()
    candidate_sorted = list(candidate_df.sort_values("score", ascending=False)[STOCK_COL])
    industry_map = dict(zip(candidate_df[STOCK_COL], candidate_df[INDUSTRY_COL].fillna("UNKNOWN").astype(str)))
    targets = build_industry_neutral_targets(
        candidate_sorted,
        industry_map,
        STOCK_NUM,
        max(1, int(np.floor(STOCK_NUM * INDUSTRY_CAP_RATIO))),
    )
    return targets, candidate_stocks


def calc_portfolio_return_from_map(ret_map, stocks):
    vals = []
    for stock in stocks:
        v = ret_map.get(stock, np.nan)
        if not pd.isnull(v):
            vals.append(float(v))
    return float(np.mean(vals)) if vals else np.nan


def evaluate_trained_model(df_all, meta_row, trained, feature_cols):
    spec = {
        "tag": meta_row["tag"],
        "train_start": meta_row["train_start"],
        "train_end": meta_row["train_end"],
        "test_start": meta_row["test_start"],
    }
    test_df = make_test_df(df_all, spec)
    if test_df.empty:
        return pd.DataFrame(), pd.DataFrame()
    test_df = test_df.copy()
    test_df["score"] = score_with_model(test_df, trained["model"], feature_cols, trained["fill_values"])
    test_df["score_rank_pct"] = test_df.groupby(DATE_COL)["score"].rank(pct=True)
    test_df["realized_rank_pct"] = test_df.groupby(DATE_COL)[TARGET_COL].rank(pct=True)

    rows = []
    panel_parts = []
    groups = list(test_df.groupby(DATE_COL))
    for rebalance_date, month_df in progress_iter(groups, total=len(groups), desc="monthly eval %s %s" % (meta_row["feature_variant"], meta_row["tag"])):
        month_df = month_df.dropna(subset=[TARGET_COL, "score"]).copy()
        if month_df.empty:
            continue
        targets, candidate_top30 = select_model_targets(month_df)
        ret_map = dict(zip(month_df[STOCK_COL], pd.to_numeric(month_df[TARGET_COL], errors="coerce")))
        target_ret = calc_portfolio_return_from_map(ret_map, targets)
        actual_top6 = list(month_df.sort_values(TARGET_COL, ascending=False)[STOCK_COL].head(REALIZED_TOP_N))
        actual_top20 = set(list(month_df.sort_values(TARGET_COL, ascending=False)[STOCK_COL].head(REALIZED_TOP20_N)))
        actual_top6_ret = calc_portfolio_return_from_map(ret_map, actual_top6)
        selected_rows = month_df.set_index(STOCK_COL).reindex(targets)
        hit_top6 = len(set(targets).intersection(set(actual_top6)))
        rows.append({
            "model_id": meta_row["model_id"],
            "feature_variant": meta_row["feature_variant"],
            "tag": meta_row["tag"],
            "train_start": meta_row["train_start"],
            "train_end": meta_row["train_end"],
            "test_start": meta_row["test_start"],
            "rebalance_date": rebalance_date,
            "target_count": len(targets),
            "candidate_count": len(candidate_top30),
            "target_ret": target_ret,
            "actual_top6_ret": actual_top6_ret,
            "oracle_gap": actual_top6_ret - target_ret,
            "rank_ic": safe_rank_ic(month_df["score"], month_df[TARGET_COL]),
            "target_avg_rank": float(selected_rows["realized_rank_pct"].mean()),
            "target_worst_rank": float(selected_rows["realized_rank_pct"].min()),
            "hit_top6": int(hit_top6),
            "fn_top6_count": int(max(0, REALIZED_TOP_N - hit_top6)),
            "fp_top6_count": int(max(0, len(targets) - hit_top6)),
            "fp_not_top20_count": int(len([s for s in targets if s not in actual_top20])),
            "targets": ",".join(targets),
            "candidate_top30": ",".join(candidate_top30),
            "actual_top6": ",".join(actual_top6),
        })
        month_df["model_id"] = meta_row["model_id"]
        month_df["feature_variant"] = meta_row["feature_variant"]
        month_df["tag"] = meta_row["tag"]
        panel_parts.append(month_df)
    monthly_df = pd.DataFrame(rows)
    score_panel_df = pd.concat(panel_parts, ignore_index=True) if panel_parts else pd.DataFrame()
    return monthly_df, score_panel_df


## 同约束随机 top6

这个指标回答的是：在同一个月份、同一个股票池、同样 top6 行业约束下，模型组合相对随机组合处于什么分位。它比单纯 `avg_rank` 更贴近实际使用场景。


In [ ]:
def encode_industries(industry_values):
    codes = []
    mapping = {}
    next_code = 0
    for x in industry_values:
        key = str(x) if not pd.isnull(x) else "UNKNOWN"
        if key == "UNKNOWN":
            codes.append(-1)
            continue
        if key not in mapping:
            mapping[key] = next_code
            next_code += 1
        codes.append(mapping[key])
    return np.asarray(codes, dtype=np.int32), len(mapping)


def select_random_indices_industry_neutral(perm, industry_codes, target_num, max_per_industry, known_industry_count):
    if len(perm) <= target_num or known_industry_count < 3:
        return perm[:min(target_num, len(perm))]

    selected = []
    selected_flag = np.zeros(len(industry_codes), dtype=np.bool_)
    industry_count = {}

    for idx in perm:
        ind = int(industry_codes[idx])
        if ind < 0:
            continue
        if industry_count.get(ind, 0) == 0:
            selected.append(idx)
            selected_flag[idx] = True
            industry_count[ind] = 1
            if len(selected) >= target_num:
                return np.asarray(selected, dtype=np.int32)

    for idx in perm:
        if selected_flag[idx]:
            continue
        ind = int(industry_codes[idx])
        if ind < 0:
            continue
        cnt = industry_count.get(ind, 0)
        if cnt < max_per_industry:
            selected.append(idx)
            selected_flag[idx] = True
            industry_count[ind] = cnt + 1
            if len(selected) >= target_num:
                return np.asarray(selected, dtype=np.int32)

    for idx in perm:
        if not selected_flag[idx]:
            selected.append(idx)
            if len(selected) >= target_num:
                break
    return np.asarray(selected, dtype=np.int32)


def random_distribution_for_month_fast(month_df, sim_n, seed_key):
    rets = pd.to_numeric(month_df[TARGET_COL], errors="coerce").replace([np.inf, -np.inf], np.nan).values.astype(float)
    valid = np.isfinite(rets)
    rets = rets[valid]
    industries = month_df.loc[valid, INDUSTRY_COL].fillna("UNKNOWN").astype(str).values
    n = len(rets)
    if n == 0:
        return np.asarray([], dtype=float)
    industry_codes, known_industry_count = encode_industries(industries)
    max_per_industry = max(1, int(np.floor(STOCK_NUM * INDUSTRY_CAP_RATIO)))
    rng = np.random.RandomState(seed_key)
    out = np.empty(int(sim_n), dtype=float)
    base_idx = np.arange(n, dtype=np.int32)
    for i in range(int(sim_n)):
        perm = rng.permutation(base_idx)
        picked = select_random_indices_industry_neutral(
            perm,
            industry_codes,
            STOCK_NUM,
            max_per_industry,
            known_industry_count,
        )
        out[i] = float(np.nanmean(rets[picked])) if len(picked) else np.nan
    return out


def stable_text_seed(text):
    text = str(text)
    total = 0
    for i, ch in enumerate(text):
        total += (i + 1) * ord(ch)
    return int(total % 100000)


def add_random_baseline(monthly_df, score_panel_df):
    rows = []
    if monthly_df.empty:
        return pd.DataFrame()
    total = len(monthly_df)
    iterator = progress_iter(monthly_df.iterrows(), total=total, desc="random baseline")
    for _, row in iterator:
        model_id = str(row["model_id"])
        dt = pd.Timestamp(row["rebalance_date"])
        month_df = score_panel_df[(score_panel_df["model_id"].astype(str) == model_id) & (score_panel_df[DATE_COL] == dt)].copy()
        month_df = month_df.dropna(subset=[TARGET_COL, "score"])
        if month_df.empty:
            continue
        target_ret = float(row["target_ret"])
        seed_key = int(pd.Timestamp(dt).strftime("%Y%m%d")) + stable_text_seed(model_id)
        rand = random_distribution_for_month_fast(month_df, RANDOM_SIM_N, seed_key)
        rand = rand[np.isfinite(rand)]
        if len(rand) == 0 or pd.isnull(target_ret):
            continue
        out = dict(row)
        out.update({
            "random_percentile": float((rand <= target_ret).mean()),
            "random_mean": float(np.nanmean(rand)),
            "random_median": float(np.nanmedian(rand)),
            "random_p10": float(np.nanpercentile(rand, 10)),
            "random_p90": float(np.nanpercentile(rand, 90)),
        })
        rows.append(out)
    return pd.DataFrame(rows)


## FP/FN 错例与特征归因

- FN：真实 top6 但模型没选上
- FP：模型选上但不是真实 top6
- FP_not_top20：模型选上但甚至不在真实 top20，优先关注


In [ ]:
def parse_targets(x):
    if pd.isnull(x):
        return []
    return [s.strip() for s in str(x).split(",") if s.strip()]


def build_fp_fn_cases(score_panel_df, random_monthly_df):
    rows = []
    if random_monthly_df.empty:
        return pd.DataFrame()
    for _, row in progress_iter(random_monthly_df.iterrows(), total=len(random_monthly_df), desc="build FP/FN cases"):
        model_id = str(row["model_id"])
        dt = pd.Timestamp(row["rebalance_date"])
        month_df = score_panel_df[(score_panel_df["model_id"].astype(str) == model_id) & (score_panel_df[DATE_COL] == dt)].copy()
        month_df = month_df.dropna(subset=[TARGET_COL, "score"])
        if month_df.empty:
            continue
        target_set = set(parse_targets(row["targets"]))
        actual_top6 = set(parse_targets(row["actual_top6"]))
        actual_top20 = set(list(month_df.sort_values(TARGET_COL, ascending=False)[STOCK_COL].head(REALIZED_TOP20_N)))
        case_defs = []
        for stock in sorted(actual_top6 - target_set):
            case_defs.append((stock, "FN_top6"))
        for stock in sorted(target_set - actual_top6):
            case_defs.append((stock, "FP_top6"))
            if stock not in actual_top20:
                case_defs.append((stock, "FP_not_top20"))
        mindex = month_df.set_index(STOCK_COL)
        for stock, case_type in case_defs:
            if stock not in mindex.index:
                continue
            r = mindex.loc[stock]
            rows.append({
                "model_id": model_id,
                "feature_variant": row["feature_variant"],
                "tag": row["tag"],
                "rebalance_date": dt,
                "case_type": case_type,
                "stock": stock,
                "industry_bucket": r.get(INDUSTRY_COL, "UNKNOWN"),
                "alpha_1m": float(r[TARGET_COL]),
                "model_score": float(r["score"]),
                "score_rank_pct": float(r["score_rank_pct"]),
                "realized_rank_pct": float(r["realized_rank_pct"]),
                "target_ret": row["target_ret"],
                "random_percentile": row["random_percentile"],
                "target_avg_rank": row["target_avg_rank"],
                "target_worst_rank": row["target_worst_rank"],
            })
    return pd.DataFrame(rows)


def get_top_gain_features(trained, feature_cols, top_n):
    model = trained.get("model", None)
    if model is None or not feature_cols:
        return []
    try:
        gain = model.feature_importance(importance_type="gain")
    except Exception:
        gain = model.feature_importance()
    imp = pd.DataFrame({"feature": feature_cols, "gain_importance": gain})
    imp = imp.sort_values("gain_importance", ascending=False)
    return list(imp["feature"].head(int(top_n)))


def feature_favorable_rank(month_df, feature, score_col="score"):
    s = pd.to_numeric(month_df[feature], errors="coerce").replace([np.inf, -np.inf], np.nan)
    score = pd.to_numeric(month_df[score_col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = pd.DataFrame({"x": s, "score": score}).dropna()
    if valid.empty or valid["x"].nunique() <= 1:
        return pd.Series(index=month_df.index, dtype=float), np.nan
    corr = valid["x"].corr(valid["score"], method="spearman")
    raw_rank = s.rank(pct=True)
    if pd.isnull(corr):
        fav = raw_rank
    elif corr >= 0:
        fav = raw_rank
    else:
        fav = 1.0 - raw_rank
    return fav, corr


def build_feature_case_attribution(score_panel_df, case_df, random_monthly_df, trained_map, feature_cols_map):
    rows = []
    if case_df.empty:
        return pd.DataFrame()
    model_ids = sorted(case_df["model_id"].astype(str).unique())
    for model_id in progress_iter(model_ids, total=len(model_ids), desc="attribute models"):
        trained = trained_map.get(model_id)
        feature_cols = feature_cols_map.get(model_id, [])
        if trained is None:
            continue
        top_features = get_top_gain_features(trained, feature_cols, ATTR_TOP_FEATURE_N)
        model_cases = case_df[case_df["model_id"].astype(str) == model_id].copy()
        month_groups = list(model_cases.groupby("rebalance_date"))
        for dt, month_cases in progress_iter(month_groups, total=len(month_groups), desc="attribute months"):
            dt = pd.Timestamp(dt)
            month_df = score_panel_df[(score_panel_df["model_id"].astype(str) == model_id) & (score_panel_df[DATE_COL] == dt)].copy()
            month_df = month_df.dropna(subset=[TARGET_COL, "score"])
            if month_df.empty:
                continue
            target_match = month_cases[month_cases["case_type"].str.startswith("FP")]
            targets = list(target_match["stock"].unique())
            random_match = random_monthly_df[(random_monthly_df["model_id"].astype(str) == model_id) & (random_monthly_df["rebalance_date"] == dt)]
            if not random_match.empty:
                targets = parse_targets(random_match.iloc[0]["targets"])
            for feature in top_features:
                if feature not in month_df.columns:
                    continue
                fav, corr = feature_favorable_rank(month_df, feature)
                month_df["_fav"] = fav
                selected_fav_median = float(month_df[month_df[STOCK_COL].isin(targets)]["_fav"].median()) if len(targets) else np.nan
                universe_fav_median = float(month_df["_fav"].median())
                fav_by_stock = month_df.set_index(STOCK_COL)["_fav"]
                val_by_stock = month_df.set_index(STOCK_COL)[feature]
                for _, case in month_cases.iterrows():
                    stock = case["stock"]
                    if stock not in fav_by_stock.index:
                        continue
                    case_fav = fav_by_stock.loc[stock]
                    if pd.isnull(case_fav):
                        continue
                    gap_vs_selected = case_fav - selected_fav_median if not pd.isnull(selected_fav_median) else np.nan
                    gap_vs_universe = case_fav - universe_fav_median if not pd.isnull(universe_fav_median) else np.nan
                    rows.append({
                        "model_id": model_id,
                        "feature_variant": case["feature_variant"],
                        "tag": case["tag"],
                        "rebalance_date": dt,
                        "case_type": case["case_type"],
                        "stock": stock,
                        "feature": feature,
                        "feature_value": val_by_stock.loc[stock],
                        "feature_score_corr": corr,
                        "case_favorable_rank": case_fav,
                        "selected_median_favorable_rank": selected_fav_median,
                        "universe_median_favorable_rank": universe_fav_median,
                        "gap_vs_selected": gap_vs_selected,
                        "gap_vs_universe": gap_vs_universe,
                        "model_score_rank": case["score_rank_pct"],
                        "realized_rank": case["realized_rank_pct"],
                    })
    return pd.DataFrame(rows)


## 批量运行实验

默认 3 个 feature variant × 2 个训练截止日，共 6 个模型。每个模型都会导出唯一命名 pkl，便于你后续逐个上传到聚宽回测。


In [ ]:
manifest_rows = []
monthly_parts = []
score_panel_parts = []
trained_map = {}
feature_cols_map = {}

jobs = []
for variant in FEATURE_VARIANTS:
    for spec in TRAIN_WINDOW_SPECS:
        jobs.append((variant, spec))

for variant, spec in progress_iter(jobs, total=len(jobs), desc="train variants"):
    spec_norm = normalize_train_spec(spec)
    print("\ntrain", variant["feature_variant"], spec_norm["tag"], spec_norm["train_start"].date(), spec_norm["train_end"].date())
    meta_row, trained, feature_cols = train_one_variant_spec(df_all, variant, spec_norm)
    manifest_rows.append(meta_row)
    trained_map[meta_row["model_id"]] = trained
    feature_cols_map[meta_row["model_id"]] = list(feature_cols)
    monthly_df, score_panel_df = evaluate_trained_model(df_all, meta_row, trained, feature_cols)
    if not monthly_df.empty:
        monthly_parts.append(monthly_df)
    if not score_panel_df.empty:
        score_panel_parts.append(score_panel_df)
    gc.collect()

manifest_df = pd.DataFrame(manifest_rows)
monthly_df = pd.concat(monthly_parts, ignore_index=True) if monthly_parts else pd.DataFrame()
score_panel_df = pd.concat(score_panel_parts, ignore_index=True) if score_panel_parts else pd.DataFrame()

print("manifest:", manifest_df.shape)
print("monthly:", monthly_df.shape)
print("score panel:", score_panel_df.shape)
display(manifest_df[["feature_variant", "tag", "train_start", "train_end", "feature_count", "train_rank_ic", "diag_rank_ic", "model_file"]])
display(monthly_df.head())


## 随机基线、FP/FN 和归因


In [ ]:
random_monthly_df = add_random_baseline(monthly_df, score_panel_df)
print("random_monthly_df:", random_monthly_df.shape)
display(random_monthly_df.head())

case_df = build_fp_fn_cases(score_panel_df, random_monthly_df)
print("case_df:", case_df.shape)
display(case_df.head(20))

feature_case_df = build_feature_case_attribution(score_panel_df, case_df, random_monthly_df, trained_map, feature_cols_map)
print("feature_case_df:", feature_case_df.shape)
display(feature_case_df.head(20))


## 汇总表

核心看三张表：

1. `v58_model_summary.csv`：模型自身训练、特征数、RankIC
2. `v58_random_summary.csv`：真实 top6 应用场景下，相对随机约束组合的分位
3. `v58_feature_failure_summary.csv`：FN/FP 主要由哪些特征驱动


In [ ]:
def build_monthly_summary(random_monthly_df):
    rows = []
    if random_monthly_df.empty:
        return pd.DataFrame()
    group_cols = ["feature_variant", "tag", "model_id", "train_start", "train_end", "test_start"]
    for keys, gdf in random_monthly_df.groupby(group_cols):
        ret_sum = summarize_return_series(gdf["target_ret"])
        rand_sum = summarize_return_series(gdf["random_mean"])
        ric = gdf["rank_ic"].replace([np.inf, -np.inf], np.nan).dropna()
        rows.append({
            "feature_variant": keys[0],
            "tag": keys[1],
            "model_id": keys[2],
            "train_start": keys[3],
            "train_end": keys[4],
            "test_start": keys[5],
            "months": ret_sum["months"],
            "target_cum_ret": ret_sum["cum_ret"],
            "target_mean_ret": ret_sum["mean_ret"],
            "target_win_rate": ret_sum["win_rate"],
            "target_max_drawdown": ret_sum["max_drawdown"],
            "random_mean_cum_ret": rand_sum["cum_ret"],
            "random_mean_ret": rand_sum["mean_ret"],
            "mean_random_percentile": float(gdf["random_percentile"].mean()),
            "median_random_percentile": float(gdf["random_percentile"].median()),
            "pct_months_above_random_median": float((gdf["random_percentile"] > 0.50).mean()),
            "pct_months_top_quartile": float((gdf["random_percentile"] > 0.75).mean()),
            "pct_months_bottom_quartile": float((gdf["random_percentile"] < 0.25).mean()),
            "avg_hit_top6": float(gdf["hit_top6"].mean()),
            "avg_fn_top6_count": float(gdf["fn_top6_count"].mean()),
            "avg_fp_top6_count": float(gdf["fp_top6_count"].mean()),
            "avg_fp_not_top20_count": float(gdf["fp_not_top20_count"].mean()),
            "avg_oracle_gap": float(gdf["oracle_gap"].mean()),
            "avg_target_rank": float(gdf["target_avg_rank"].mean()),
            "rank_ic_mean": float(ric.mean()) if len(ric) else np.nan,
            "rank_ic_ir": float(ric.mean() / ric.std()) if len(ric) > 1 and ric.std() > 0 else np.nan,
            "drop_top1_cum_ret": float((1.0 + gdf["target_ret"].drop(gdf["target_ret"].idxmax())).prod() - 1.0) if len(gdf) > 1 else np.nan,
        })
    out = pd.DataFrame(rows)
    return out.sort_values(["tag", "target_cum_ret"], ascending=[True, False])


def build_case_summary(case_df):
    if case_df.empty:
        return pd.DataFrame()
    rows = []
    for keys, gdf in case_df.groupby(["feature_variant", "tag", "case_type"]):
        rows.append({
            "feature_variant": keys[0],
            "tag": keys[1],
            "case_type": keys[2],
            "cases": int(len(gdf)),
            "months": int(gdf["rebalance_date"].nunique()),
            "mean_alpha": float(gdf["alpha_1m"].mean()),
            "mean_score_rank": float(gdf["score_rank_pct"].mean()),
            "mean_realized_rank": float(gdf["realized_rank_pct"].mean()),
        })
    return pd.DataFrame(rows).sort_values(["tag", "case_type", "cases"], ascending=[True, True, False])


def build_feature_failure_summary(feature_case_df):
    if feature_case_df.empty:
        return pd.DataFrame()
    parts = []
    for keys, gdf in feature_case_df.groupby(["feature_variant", "tag", "case_type", "feature"]):
        feature_variant, tag, case_type, feature = keys
        if str(case_type).startswith("FN"):
            severity = -gdf["gap_vs_selected"]
        else:
            severity = gdf["case_favorable_rank"]
        parts.append({
            "feature_variant": feature_variant,
            "tag": tag,
            "case_type": case_type,
            "feature": feature,
            "cases": int(len(gdf)),
            "avg_case_favorable_rank": float(gdf["case_favorable_rank"].mean()),
            "avg_gap_vs_selected": float(gdf["gap_vs_selected"].mean()),
            "avg_gap_vs_universe": float(gdf["gap_vs_universe"].mean()),
            "avg_feature_score_corr": float(gdf["feature_score_corr"].mean()),
            "severity_score": float(severity.mean()),
        })
    out = pd.DataFrame(parts)
    return out.sort_values(["tag", "case_type", "severity_score"], ascending=[True, True, False])


random_summary_df = build_monthly_summary(random_monthly_df)
case_summary_df = build_case_summary(case_df)
feature_failure_summary_df = build_feature_failure_summary(feature_case_df)

summary_cols = [
    "model_id", "feature_variant", "tag", "months", "target_cum_ret", "target_mean_ret",
    "target_win_rate", "target_max_drawdown", "mean_random_percentile",
    "median_random_percentile", "pct_months_above_random_median", "avg_fp_not_top20_count",
    "avg_oracle_gap", "avg_target_rank", "rank_ic_mean", "drop_top1_cum_ret",
]
model_summary_df = manifest_df.merge(
    random_summary_df[[c for c in summary_cols if c in random_summary_df.columns]],
    on=["model_id", "feature_variant", "tag"],
    how="left",
)

print("model summary")
display(model_summary_df[[c for c in [
    "feature_variant", "tag", "train_start", "train_end", "feature_count",
    "target_cum_ret", "target_max_drawdown", "mean_random_percentile",
    "avg_fp_not_top20_count", "diag_rank_ic", "model_file"
] if c in model_summary_df.columns]])

print("random summary")
display(random_summary_df)
print("case summary")
display(case_summary_df)
print("feature failure summary")
display(feature_failure_summary_df.head(100))


## 导出


In [ ]:
variant_manifest_df.to_csv(os.path.join(OUT_DIR, "v58_feature_variant_manifest.csv"), index=False)
manifest_df.to_csv(os.path.join(OUT_DIR, "v58_model_manifest.csv"), index=False)
model_summary_df.to_csv(os.path.join(OUT_DIR, "v58_model_summary.csv"), index=False)
monthly_df.to_csv(os.path.join(OUT_DIR, "v58_monthly.csv"), index=False)
random_monthly_df.to_csv(os.path.join(OUT_DIR, "v58_random_monthly.csv"), index=False)
random_summary_df.to_csv(os.path.join(OUT_DIR, "v58_random_summary.csv"), index=False)
case_df.to_csv(os.path.join(OUT_DIR, "v58_fp_fn_cases.csv"), index=False)
case_summary_df.to_csv(os.path.join(OUT_DIR, "v58_fp_fn_summary.csv"), index=False)
feature_case_df.to_csv(os.path.join(OUT_DIR, "v58_feature_attribution.csv"), index=False)
feature_failure_summary_df.to_csv(os.path.join(OUT_DIR, "v58_feature_failure_summary.csv"), index=False)

print("saved outputs to:", OUT_DIR)
print("model pkls exported:", bool(EXPORT_MODELS))
if not random_summary_df.empty:
    display(random_summary_df.sort_values(["tag", "target_cum_ret"], ascending=[True, False]))


## 结论阅读模板

跑完后建议按这个顺序看：

1. 同一个 `tag` 下比较 `target_cum_ret / max_drawdown / mean_random_percentile / avg_fp_not_top20_count`。
2. 如果 `full` 收益最高但 `fp_not_top20` 也显著最高，说明技术价格特征可能提高攻击性但带来错选风险。
3. 如果 `light_technical_price` 接近或超过 `full`，优先考虑它作为下一轮回测候选，因为它保留了一部分月频风格信号，同时降低了细粒度技术特征过拟合风险。
4. 如果 `no_technical_price` 明显弱，说明完全去掉技术/价格族会丢掉有效边际；如果它回撤明显更低，则可以作为防守/降风险候选。
5. 最后看 `v58_feature_failure_summary.csv`：如果 FP_not_top20 长期由某几个技术特征驱动，可以在下一轮只删这些特征，而不是整族删除。
